In [0]:
# Gold layer transformations - creating business-ready aggregated tables
# for analytics dashboards and reporting

import pyspark.sql.functions as F
from pyspark.sql.window import Window

# Load silver data
silver_table = "finops_catalog.focus_billing_schema.silver_focus_billing"
silver_df = spark.table(silver_table)

print(f"Loaded {silver_df.count():,} records from {silver_table}")
print("Creating gold aggregations...\n")

In [0]:
# Daily cost by service - for monitoring daily spend and trends

print("Creating daily cost by service...")

gold_daily_service = silver_df.withColumn(
    "charge_date", F.to_date("ChargePeriodStart")
).withColumn(
    "charge_year", F.year("ChargePeriodStart")
).withColumn(
    "charge_month", F.month("ChargePeriodStart")
).withColumn(
    "charge_day_of_week", F.dayofweek("ChargePeriodStart")
).withColumn(
    "discount_amount", F.col("ListCost") - F.col("EffectiveCost")
).withColumn(
    "discount_percentage",
    F.when(F.col("ListCost") > 0,
           ((F.col("ListCost") - F.col("EffectiveCost")) / F.col("ListCost")) * 100)
     .otherwise(0)
).withColumn(
    "is_commitment_discount_applied",
    F.when(F.col("CommitmentDiscountId").isNotNull(), True).otherwise(False)
).withColumn(
    "service_family",
    F.when(F.col("ServiceName").contains("Compute"), "Compute")
     .when(F.col("ServiceName").contains("Storage") | 
           F.col("ServiceName").contains("S3"), "Storage")
     .when(F.col("ServiceName").contains("Database") | 
           F.col("ServiceName").contains("RDS"), "Database")
     .when(F.col("ServiceName").contains("Network") | 
           F.col("ServiceName").contains("VPC") |
           F.col("ServiceName").contains("CloudFront"), "Networking")
     .when(F.col("ServiceName").contains("Lambda"), "Serverless")
     .otherwise("Other")
).groupBy(
    "charge_date",
    "charge_year",
    "charge_month",
    "charge_day_of_week",
    "ServiceName",
    "service_family",
    "BillingAccountName",
    "RegionName"
).agg(
    F.sum("BilledCost").alias("total_billed_cost"),
    F.sum("EffectiveCost").alias("total_effective_cost"),
    F.sum("ListCost").alias("total_list_cost"),
    F.sum("discount_amount").alias("total_savings"),
    F.avg("discount_percentage").alias("avg_discount_pct"),
    F.sum(F.expr("try_cast(ConsumedQuantity as double)")).alias("total_quantity_consumed"),
    F.count("*").alias("record_count"),
    F.countDistinct("ResourceId").alias("unique_resources"),
    F.sum(F.when(F.col("is_commitment_discount_applied"), 1).otherwise(0)).alias("commitment_discount_records")
).withColumn(
    "commitment_coverage_pct",
    (F.col("commitment_discount_records") / F.col("record_count")) * 100
).withColumn(
    "avg_cost_per_resource",
    F.when(F.col("unique_resources") > 0, 
           F.col("total_billed_cost") / F.col("unique_resources"))
     .otherwise(None)
).withColumn(
    "gold_created_at",
    F.current_timestamp()
)

gold_daily_service_table = "finops_catalog.focus_billing_schema.gold_daily_cost_by_service"
gold_daily_service.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(gold_daily_service_table)

record_count = spark.table(gold_daily_service_table).count()
print(f"Created {gold_daily_service_table} with {record_count:,} records")

In [0]:
# Monthly cost summary with month-over-month comparisons

print("Creating monthly cost summary...")

gold_monthly = silver_df.withColumn(
    "charge_year", F.year("ChargePeriodStart")
).withColumn(
    "charge_month", F.month("ChargePeriodStart")
).withColumn(
    "charge_year_month", F.date_format("ChargePeriodStart", "yyyy-MM")
).withColumn(
    "discount_amount", F.col("ListCost") - F.col("EffectiveCost")
).withColumn(
    "discount_percentage",
    F.when(F.col("ListCost") > 0,
           ((F.col("ListCost") - F.col("EffectiveCost")) / F.col("ListCost")) * 100)
     .otherwise(0)
).withColumn(
    "is_commitment_discount_applied",
    F.when(F.col("CommitmentDiscountId").isNotNull(), True).otherwise(False)
).withColumn(
    "is_on_demand",
    F.when(F.col("CommitmentDiscountId").isNull(), True).otherwise(False)
).withColumn(
    "cost_category",
    F.when(F.col("BilledCost") == 0, "Free/No Charge")
     .when(F.col("BilledCost") >= 1000, "Very Large (>=$1K)")
     .when(F.col("BilledCost") >= 100, "Large ($100-$1K)")
     .when(F.col("BilledCost") >= 10, "Medium ($10-$100)")
     .when(F.col("BilledCost") >= 1, "Small ($1-$10)")
     .otherwise("Micro (<$1)")
).groupBy(
    "charge_year_month",
    "charge_year",
    "charge_month",
    "BillingAccountName",
    "BillingAccountId"
).agg(
    F.sum("BilledCost").alias("total_monthly_cost"),
    F.sum("EffectiveCost").alias("total_effective_cost"),
    F.sum("discount_amount").alias("total_monthly_savings"),
    F.avg("discount_percentage").alias("avg_discount_pct"),
    F.count("*").alias("total_records"),
    F.countDistinct("ServiceName").alias("unique_services_used"),
    F.countDistinct("ResourceId").alias("unique_resources"),
    F.countDistinct("RegionId").alias("unique_regions"),
    F.sum(F.when(F.col("cost_category") == "Free/No Charge", F.col("BilledCost")).otherwise(0)).alias("free_tier_usage"),
    F.sum(F.when(F.col("cost_category") == "Very Large (>=$1K)", F.col("BilledCost")).otherwise(0)).alias("high_cost_items"),
    F.sum(F.when(F.col("is_commitment_discount_applied"), F.col("BilledCost")).otherwise(0)).alias("commitment_covered_cost"),
    F.sum(F.when(F.col("is_on_demand"), F.col("BilledCost")).otherwise(0)).alias("on_demand_cost")
).withColumn(
    "commitment_coverage_pct",
    F.when(F.col("total_monthly_cost") > 0,
           (F.col("commitment_covered_cost") / F.col("total_monthly_cost")) * 100)
     .otherwise(0)
).withColumn(
    "on_demand_pct",
    F.when(F.col("total_monthly_cost") > 0,
           (F.col("on_demand_cost") / F.col("total_monthly_cost")) * 100)
     .otherwise(0)
).withColumn(
    "gold_created_at",
    F.current_timestamp()
)

# Add month-over-month comparison
window_spec = Window.partitionBy("BillingAccountId").orderBy("charge_year_month")

gold_monthly = gold_monthly.withColumn(
    "previous_month_cost",
    F.lag("total_monthly_cost", 1).over(window_spec)
).withColumn(
    "month_over_month_change",
    F.col("total_monthly_cost") - F.col("previous_month_cost")
).withColumn(
    "month_over_month_change_pct",
    F.when(F.col("previous_month_cost") > 0,
           ((F.col("total_monthly_cost") - F.col("previous_month_cost")) / F.col("previous_month_cost")) * 100)
     .otherwise(None)
)

gold_monthly_table = "finops_catalog.focus_billing_schema.gold_monthly_cost_summary"
gold_monthly.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(gold_monthly_table)

record_count = spark.table(gold_monthly_table).count()
print(f"Created {gold_monthly_table} with {record_count:,} records")

In [0]:
# Resource cost ranking - identify top cost drivers

print("Creating resource cost ranking...")

gold_resources = silver_df.withColumn(
    "charge_date", F.to_date("ChargePeriodStart")
).withColumn(
    "discount_amount", F.col("ListCost") - F.col("EffectiveCost")
).withColumn(
    "effective_unit_price",
    F.when(F.expr("try_cast(ConsumedQuantity as double)") > 0,
           F.col("EffectiveCost") / F.expr("try_cast(ConsumedQuantity as double)"))
     .otherwise(None)
).withColumn(
    "is_commitment_discount_applied",
    F.when(F.col("CommitmentDiscountId").isNotNull(), True).otherwise(False)
).withColumn(
    "is_on_demand",
    F.when(F.col("CommitmentDiscountId").isNull(), True).otherwise(False)
).withColumn(
    "service_family",
    F.when(F.col("ServiceName").contains("Compute"), "Compute")
     .when(F.col("ServiceName").contains("Storage") | 
           F.col("ServiceName").contains("S3"), "Storage")
     .when(F.col("ServiceName").contains("Database") | 
           F.col("ServiceName").contains("RDS"), "Database")
     .when(F.col("ServiceName").contains("Network") | 
           F.col("ServiceName").contains("VPC") |
           F.col("ServiceName").contains("CloudFront"), "Networking")
     .when(F.col("ServiceName").contains("Lambda"), "Serverless")
     .otherwise("Other")
).filter(
    F.col("ResourceId").isNotNull()
).groupBy(
    "ResourceId",
    "ResourceName",
    "ResourceType",
    "ServiceName",
    "service_family",
    "RegionName",
    "AvailabilityZone"
).agg(
    F.sum("BilledCost").alias("total_resource_cost"),
    F.sum("EffectiveCost").alias("total_effective_cost"),
    F.sum("discount_amount").alias("total_savings"),
    F.avg("effective_unit_price").alias("avg_unit_price"),
    F.sum(F.expr("try_cast(ConsumedQuantity as double)")).alias("total_quantity_consumed"),
    F.count("*").alias("charge_record_count"),
    F.min("charge_date").alias("first_charge_date"),
    F.max("charge_date").alias("last_charge_date"),
    F.countDistinct("charge_date").alias("days_active"),
    F.sum(F.when(F.col("is_commitment_discount_applied"), 1).otherwise(0)).alias("commitment_discount_count"),
    F.sum(F.when(F.col("is_on_demand"), F.col("BilledCost")).otherwise(0)).alias("on_demand_cost")
).withColumn(
    "avg_daily_cost",
    F.when(F.col("days_active") > 0,
           F.col("total_resource_cost") / F.col("days_active"))
     .otherwise(None)
).withColumn(
    "commitment_coverage_pct",
    F.when(F.col("charge_record_count") > 0,
           (F.col("commitment_discount_count") / F.col("charge_record_count")) * 100)
     .otherwise(0)
).withColumn(
    "optimization_opportunity",
    F.when((F.col("on_demand_cost") > 100) & (F.col("commitment_coverage_pct") < 50), "High")
     .when((F.col("on_demand_cost") > 50) & (F.col("commitment_coverage_pct") < 70), "Medium")
     .when(F.col("on_demand_cost") > 10, "Low")
     .otherwise("None")
)

# Rank resources by cost within each service family
window_rank = Window.partitionBy("service_family").orderBy(F.desc("total_resource_cost"))

gold_resources = gold_resources.withColumn(
    "cost_rank_in_service_family",
    F.rank().over(window_rank)
).withColumn(
    "gold_created_at",
    F.current_timestamp()
)

gold_resources_table = "finops_catalog.focus_billing_schema.gold_resource_cost_ranking"
gold_resources.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(gold_resources_table)

record_count = spark.table(gold_resources_table).count()
print(f"Created {gold_resources_table} with {record_count:,} resources")

In [0]:
# Commitment discount effectiveness - track RI/Savings Plan utilization

print("Creating commitment effectiveness...")

gold_commitments = silver_df.withColumn(
    "charge_year_month", F.date_format("ChargePeriodStart", "yyyy-MM")
).withColumn(
    "discount_amount", F.col("ListCost") - F.col("EffectiveCost")
).withColumn(
    "discount_percentage",
    F.when(F.col("ListCost") > 0,
           ((F.col("ListCost") - F.col("EffectiveCost")) / F.col("ListCost")) * 100)
     .otherwise(0)
).groupBy(
    "charge_year_month",
    "CommitmentDiscountType",
    "CommitmentDiscountName",
    "CommitmentDiscountStatus",
    "ServiceName",
    "RegionName"
).agg(
    F.sum("BilledCost").alias("total_cost"),
    F.sum("EffectiveCost").alias("total_effective_cost"),
    F.sum("ListCost").alias("total_list_cost"),
    F.sum("discount_amount").alias("total_savings_realized"),
    F.avg("discount_percentage").alias("avg_discount_pct"),
    F.count("*").alias("charge_count"),
    F.countDistinct("ResourceId").alias("unique_resources_covered"),
    F.sum(F.expr("try_cast(ConsumedQuantity as double)")).alias("total_quantity")
).withColumn(
    "savings_per_resource",
    F.when(F.col("unique_resources_covered") > 0,
           F.col("total_savings_realized") / F.col("unique_resources_covered"))
     .otherwise(None)
).withColumn(
    "effective_savings_rate",
    F.when(F.col("total_list_cost") > 0,
           (F.col("total_savings_realized") / F.col("total_list_cost")) * 100)
     .otherwise(0)
).withColumn(
    "gold_created_at",
    F.current_timestamp()
)

gold_commitments_table = "finops_catalog.focus_billing_schema.gold_commitment_effectiveness"
gold_commitments.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(gold_commitments_table)

record_count = spark.table(gold_commitments_table).count()
print(f"Created {gold_commitments_table} with {record_count:,} records")

In [0]:
# Service family cost distribution - high-level spend by category

print("Creating service family distribution...")

gold_service_family = silver_df.withColumn(
    "charge_date", F.to_date("ChargePeriodStart")
).withColumn(
    "charge_year_month", F.date_format("ChargePeriodStart", "yyyy-MM")
).withColumn(
    "discount_amount", F.col("ListCost") - F.col("EffectiveCost")
).withColumn(
    "is_commitment_discount_applied",
    F.when(F.col("CommitmentDiscountId").isNotNull(), True).otherwise(False)
).withColumn(
    "is_on_demand",
    F.when(F.col("CommitmentDiscountId").isNull(), True).otherwise(False)
).withColumn(
    "service_family",
    F.when(F.col("ServiceName").contains("Compute"), "Compute")
     .when(F.col("ServiceName").contains("Storage") | 
           F.col("ServiceName").contains("S3"), "Storage")
     .when(F.col("ServiceName").contains("Database") | 
           F.col("ServiceName").contains("RDS"), "Database")
     .when(F.col("ServiceName").contains("Network") | 
           F.col("ServiceName").contains("VPC") |
           F.col("ServiceName").contains("CloudFront"), "Networking")
     .when(F.col("ServiceName").contains("Lambda"), "Serverless")
     .otherwise("Other")
).groupBy(
    "charge_date",
    "charge_year_month",
    "service_family",
    "BillingAccountName"
).agg(
    F.sum("BilledCost").alias("total_cost"),
    F.sum("EffectiveCost").alias("total_effective_cost"),
    F.sum("discount_amount").alias("total_savings"),
    F.count("*").alias("record_count"),
    F.countDistinct("ServiceName").alias("unique_services"),
    F.countDistinct("ResourceId").alias("unique_resources"),
    F.sum(F.when(F.col("is_on_demand"), F.col("BilledCost")).otherwise(0)).alias("on_demand_cost"),
    F.sum(F.when(F.col("is_commitment_discount_applied"), F.col("BilledCost")).otherwise(0)).alias("commitment_cost")
).withColumn(
    "on_demand_pct",
    F.when(F.col("total_cost") > 0,
           (F.col("on_demand_cost") / F.col("total_cost")) * 100)
     .otherwise(0)
).withColumn(
    "commitment_pct",
    F.when(F.col("total_cost") > 0,
           (F.col("commitment_cost") / F.col("total_cost")) * 100)
     .otherwise(0)
).withColumn(
    "avg_cost_per_service",
    F.when(F.col("unique_services") > 0,
           F.col("total_cost") / F.col("unique_services"))
     .otherwise(None)
).withColumn(
    "gold_created_at",
    F.current_timestamp()
)

# Calculate what % of daily total each service family represents
window_total = Window.partitionBy("charge_date", "BillingAccountName")

gold_service_family = gold_service_family.withColumn(
    "daily_total_cost",
    F.sum("total_cost").over(window_total)
).withColumn(
    "pct_of_daily_total",
    F.when(F.col("daily_total_cost") > 0,
           (F.col("total_cost") / F.col("daily_total_cost")) * 100)
     .otherwise(0)
)

gold_service_family_table = "finops_catalog.focus_billing_schema.gold_service_family_distribution"
gold_service_family.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(gold_service_family_table)

record_count = spark.table(gold_service_family_table).count()
print(f"Created {gold_service_family_table} with {record_count:,} records")

In [0]:
# Summary of all gold tables created

print("\n" + "="*70)
print("Gold layer complete!")
print("="*70)

gold_tables = [
    "gold_daily_cost_by_service",
    "gold_monthly_cost_summary",
    "gold_resource_cost_ranking",
    "gold_commitment_effectiveness",
    "gold_service_family_distribution"
]

print("\nTables created:")
for i, table in enumerate(gold_tables, 1):
    full_table_name = f"finops_catalog.focus_billing_schema.{table}"
    count = spark.table(full_table_name).count()
    print(f"  {i}. {table} - {count:,} records")

print("\nWhat you can do with these:")
print("  - monthly_cost_summary: great for executive reports and budget tracking")
print("  - daily_cost_by_service: monitor daily spending patterns")
print("  - resource_cost_ranking: find your top spenders and optimization opportunities")
print("  - commitment_effectiveness: see how well your RIs/Savings Plans are working")
print("  - service_family_distribution: high-level view of where money goes")

print("\nNext steps:")
print("  - Hook up your BI tool to these tables")
print("  - Set up some alerts for budget overruns")
print("  - Schedule this to run daily or weekly")
print("\nDone! Bronze -> Silver -> Gold pipeline complete.")
print("="*70)